# Condition Analysis — MSCOCO (No Ground-Truth Types)

Structured exploration of what learned conditions do in the CoSiR combiner, adapted for datasets **without** ground-truth text-type labels.

MSCOCO provides 5 captions per image, which we exploit as a structural probe in place of explicit type labels:
the **within-image condition consistency** (Section 2) and **cross-caption condition transfer** (Section 2b)
replace the type-mean and separability analyses from the impressions notebook.

The combiner here operates on the **image side** (`combine_side='img'`): it takes an image embedding
plus a condition vector and returns a modified image embedding. Retrieval is then performed by querying
text against this conditioned image gallery.

**Sections:**
1. Condition Space: Geometry & Clustering *(merged)*
2. Within-Image Condition Consistency *(the key MSCOCO diagnostic)*
3. Caption Property ↔ Condition Correlation
4. Ablation: Retrieval Quality under Different Conditions *(merged with per-sample distribution)*
5. Directional Sweep Along PCA Axes
6. Epoch Evolution *(geometry-based, no labels)*
7. Per-Image Deep Dive

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
EXPERIMENT_DIR  = "/project/CoSiR/res/CoSiR_Experiment/coco/20260504_140042_CoSiR_Experiment"
TRAIN_ANN_PATH  = "/data/SSD/coco/annotations/coco_karpathy_train.json"
IMAGE_ROOT      = "/data/SSD/coco/images"
EPOCH           = None   # None → latest checkpoint
DEVICE          = "cuda"

N_VIZ           = 15000  # conditions to sample for scatter plots
K_CLUSTER       = None   # None → auto-select via silhouette
N_SWEEP         = 15     # steps in directional sweep
N_RETRIEVE      = 5      # images shown per retrieval row
N_WITHIN_SAMPLE = 8000   # images sampled for within-image analysis per epoch
DEEP_DIVE_IMGS  = [2, 42, 137]  # test image indices to inspect in Section 7
SWEEP_QUERY_IDX = 0      # test text index used as fixed query in directional sweep
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, glob, json, importlib.util
from collections import defaultdict

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Import Combiner_new directly to avoid src/model/__init__.py pulling in cuml
_repo_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../.."))
_spec = importlib.util.spec_from_file_location("combiner", os.path.join(_repo_root, "src/model/combiner.py"))
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
Combiner_new = _mod.Combiner_new

cond_dir = os.path.join(EXPERIMENT_DIR, "condition_viz")

# ── Test-set fixed data ───────────────────────────────────────────────────────
fixed        = torch.load(os.path.join(cond_dir, "fixed_data.pt"), map_location="cpu", weights_only=False)
all_img_emb  = fixed["all_img_emb"]          # [5000, 512]
all_txt_emb  = fixed["all_txt_emb"]          # [25000, 512]
all_raw_text = fixed["all_raw_text"]         # list[25000]
image_paths  = fixed["image_paths"]          # list[5000]  — absolute paths
img2txt      = fixed["image_to_text_map"]    # [5000, 5]
txt2img      = fixed["text_to_image_map"]    # [25000]
cpi          = fixed["captions_per_image"]   # 5
n_img, n_txt = all_img_emb.shape[0], all_txt_emb.shape[0]

# ── Training conditions ───────────────────────────────────────────────────────
epoch_files = sorted(glob.glob(os.path.join(cond_dir, "epoch_*.pt")))
snap_path   = epoch_files[-1] if EPOCH is None else os.path.join(cond_dir, f"epoch_{EPOCH:04d}.pt")
snap        = torch.load(snap_path, map_location="cpu", weights_only=False)
epoch           = snap["epoch"]
label_emb_all   = snap["label_embeddings_all"]   # [566747, 16]
representatives  = snap["representatives"]         # [K, 16]
combiner_cfg     = snap["combiner_config"]
combine_side     = snap.get("combine_side", "img")
K_reps           = representatives.shape[0]
label_dim        = label_emb_all.shape[1]

assert combine_side == "img", f"This notebook is written for img-side; got {combine_side!r}"

# ── Combiner ──────────────────────────────────────────────────────────────────
combiner = Combiner_new(**combiner_cfg).to(DEVICE)
combiner.load_state_dict(snap["combiner_state_dict"])
combiner.eval()

# ── Training annotations (same order as label_emb_all) ───────────────────────
train_anns      = json.load(open(TRAIN_ANN_PATH))
train_captions  = [a["caption"].strip() for a in train_anns]
train_image_ids = [a["image_id"] for a in train_anns]
assert len(train_captions) == label_emb_all.shape[0], "Annotation / condition count mismatch"

# ── Precomputed normalised embeddings ─────────────────────────────────────────
img_n = F.normalize(all_img_emb, dim=-1)   # [5000, 512]
txt_n = F.normalize(all_txt_emb, dim=-1)   # [25000, 512]

# ── image_id → training-sample indices ───────────────────────────────────────
img_id_to_idxs: dict[str, list[int]] = defaultdict(list)
for i, img_id in enumerate(train_image_ids):
    img_id_to_idxs[img_id].append(i)
multi_cap_ids = [img_id for img_id, idxs in img_id_to_idxs.items() if len(idxs) >= 2]

print(f"Epoch {epoch} | {label_emb_all.shape[0]:,} train conditions | dim={label_dim} | {K_reps} reps")
print(f"Test : {n_img} images × {cpi} caps = {n_txt} texts")
print(f"Train: {len(train_captions):,} samples | {len(img_id_to_idxs):,} unique images")
print(f"combine_side = {combine_side!r}  → combiner modifies the IMAGE gallery")

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

@torch.no_grad()
def apply_condition(img_emb: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
    """Apply one condition to a batch of image embeddings (img-side combiner)."""
    if cond.dim() == 1:
        cond = cond.unsqueeze(0).expand(img_emb.shape[0], -1)
    return combiner(img_emb.to(DEVICE), None, cond.to(DEVICE)).cpu()


@torch.no_grad()
def retrieval_metrics(cond: torch.Tensor | None, max_queries: int = n_txt) -> dict:
    """R@1/5/10 and mean rank. cond=None → CLIP baseline (no combiner)."""
    if cond is None:
        gallery_n = img_n
    else:
        gallery_n = F.normalize(apply_condition(all_img_emb, cond), dim=-1)

    q_idx    = torch.arange(min(n_txt, max_queries))
    scores   = txt_n[q_idx] @ gallery_n.T          # [N_q, 5000]
    gt_img   = txt2img[q_idx]                       # [N_q]
    gt_score = scores[torch.arange(len(q_idx)), gt_img]
    ranks    = (scores > gt_score.unsqueeze(1)).sum(1).numpy() + 1   # [N_q]
    return {
        "mean_rank": float(ranks.mean()),
        "R@1":  float((ranks <= 1).mean() * 100),
        "R@5":  float((ranks <= 5).mean() * 100),
        "R@10": float((ranks <= 10).mean() * 100),
        "ranks": ranks,
    }


def load_image(path: str, size: int = 112) -> Image.Image:
    try:
        return Image.open(path).convert("RGB").resize((size, size))
    except Exception:
        return Image.new("RGB", (size, size), (180, 180, 180))


def show_img(ax, path, border_color=None, size=112):
    ax.imshow(load_image(path, size))
    ax.axis("off")
    if border_color:
        for sp in ax.spines.values():
            sp.set_edgecolor(border_color)
            sp.set_linewidth(3)


median_norm = float(label_emb_all.norm(dim=-1).median())
print(f"Condition dim={label_dim} | median norm={median_norm:.4f}")

---
## 1 — Condition Space: Geometry & Clustering

**What this shows.** The 16-dimensional condition space projected to 2D (PCA).
Without ground-truth type labels we use two coloring schemes:
- **Left:** condition L2-norm — do high-norm conditions cluster spatially?
- **Middle:** caption word count — does caption length predict location in condition space?
- **Right:** K-means cluster membership (K selected by silhouette score).

The clustering panel replaces the type-colored scatter from the impressions notebook.
Representative captions from each cluster give an interpretable label for what the model discovered.

In [ ]:
from sklearn.decomposition import PCA

rng = np.random.default_rng(42)
viz_idx   = rng.choice(len(label_emb_all), size=N_VIZ, replace=False)
viz_conds = label_emb_all[viz_idx].numpy()                    # [N_VIZ, 16]
viz_caps  = [train_captions[i] for i in viz_idx]

pca2 = PCA(n_components=2, random_state=42)
conds_2d = pca2.fit_transform(viz_conds)                      # [N_VIZ, 2]

norms_viz  = np.linalg.norm(viz_conds, axis=1)
wc_viz     = np.array([len(c.split()) for c in viz_caps])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc0 = axes[0].scatter(conds_2d[:, 0], conds_2d[:, 1],
                      c=norms_viz, cmap="viridis", s=4, alpha=0.4)
plt.colorbar(sc0, ax=axes[0])
axes[0].set_title("Condition norm")

sc1 = axes[1].scatter(conds_2d[:, 0], conds_2d[:, 1],
                      c=wc_viz, cmap="plasma", s=4, alpha=0.4,
                      vmax=np.percentile(wc_viz, 95))
plt.colorbar(sc1, ax=axes[1])
axes[1].set_title("Caption word count")

for ax in axes:
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")

plt.suptitle("PCA of learned conditions (training set sample)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score

# Select K via silhouette (on the N_VIZ sample)
k_candidates = [3, 4, 5, 6, 8, 10]
sil_scores = {}
for k in k_candidates:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=5)
    lbl    = km_tmp.fit_predict(viz_conds)
    sil    = silhouette_score(viz_conds, lbl, sample_size=min(N_VIZ, 5000), random_state=42)
    sil_scores[k] = sil
    print(f"  K={k:2d}  silhouette={sil:.4f}")

best_K = K_CLUSTER if K_CLUSTER is not None else max(sil_scores, key=sil_scores.get)
print(f"\nUsing K={best_K}")

# Fit final K-means on the full training set (MiniBatch for speed)
all_conds_np = label_emb_all.numpy()   # [566747, 16]
km = MiniBatchKMeans(n_clusters=best_K, random_state=42, batch_size=8192, n_init=5)
km.fit(all_conds_np)
cluster_labels_viz = km.predict(viz_conds)

COLORS = plt.cm.tab10(np.linspace(0, 0.9, best_K))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for k in range(best_K):
    mask = cluster_labels_viz == k
    axes[0].scatter(conds_2d[mask, 0], conds_2d[mask, 1],
                    s=4, alpha=0.5, color=COLORS[k], label=f"C{k}")
axes[0].legend(markerscale=3, fontsize=8)
axes[0].set_title(f"K-means clusters (K={best_K})")
axes[0].set_xlabel(f"PC1"); axes[0].set_ylabel(f"PC2")

axes[1].plot(list(sil_scores.keys()), list(sil_scores.values()), "o-")
axes[1].axvline(best_K, color="r", linestyle="--", label=f"chosen K={best_K}")
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette score")
axes[1].set_title("K selection"); axes[1].legend()
plt.tight_layout(); plt.show()

# Representative captions per cluster (8 nearest to centroid)
print("\n── Representative captions per cluster ──")
for k in range(best_K):
    center = km.cluster_centers_[k]
    dists  = np.linalg.norm(all_conds_np - center, axis=1)
    top8   = np.argsort(dists)[:8]
    wcs    = [len(train_captions[i].split()) for i in top8]
    print(f"\n[Cluster {k}]  center norm={np.linalg.norm(center):.3f}  avg word count≈{np.mean(wcs):.1f}")
    for j, idx in enumerate(top8):
        print(f"  {j+1}. {train_captions[idx]}")

---
## 2 — Within-Image Condition Consistency

**What this shows.** The key diagnostic for MSCOCO: each training image has ≈5 captions,
each with its own learned condition. If the condition encodes **image-level** information
(what the image depicts), captions of the same image should receive similar conditions.
If the condition encodes **caption-level** information (phrasing, style), conditions for
the same image should scatter no more than random pairs.

This replaces the type-separability analysis from the impressions notebook.

- **Left:** histogram of pairwise cosine similarities within an image group vs. random pairs.
- **Right (2b):** scatter — within-image caption word-count variance vs. within-image condition
  variance. Tests whether linguistically diverse caption sets produce more spread-out conditions.

In [ ]:
# Normalised training conditions
conds_np_n = all_conds_np / (np.linalg.norm(all_conds_np, axis=1, keepdims=True) + 1e-8)

# Within-image pairwise cosine similarities
within_sims = []
for img_id in multi_cap_ids:
    idxs = img_id_to_idxs[img_id]
    c    = conds_np_n[idxs]          # [n_caps, 16]
    sim  = c @ c.T                   # [n_caps, n_caps]
    n    = len(idxs)
    within_sims.extend(sim[i, j] for i in range(n) for j in range(i + 1, n))
within_arr = np.array(within_sims)

# Random (across-image) pairs as baseline
rand_a = rng.choice(len(conds_np_n), size=len(within_arr), replace=True)
rand_b = rng.choice(len(conds_np_n), size=len(within_arr), replace=True)
across_arr = (conds_np_n[rand_a] * conds_np_n[rand_b]).sum(axis=1)

print(f"Within-image pairs : {len(within_arr):,}")
print(f"  cosine sim  mean={within_arr.mean():.4f}  std={within_arr.std():.4f}  median={np.median(within_arr):.4f}")
print(f"Random pairs       : {len(across_arr):,}")
print(f"  cosine sim  mean={across_arr.mean():.4f}  std={across_arr.std():.4f}  median={np.median(across_arr):.4f}")

# Effect size (Cohen's d)
pooled_std = np.sqrt((within_arr.std()**2 + across_arr.std()**2) / 2)
cohen_d    = (within_arr.mean() - across_arr.mean()) / (pooled_std + 1e-8)
print(f"\nCohen's d = {cohen_d:.3f}  (|d|>0.8 = large effect)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 2a: histogram
axes[0].hist(within_arr, bins=80, alpha=0.7, color="#2196F3",
             label=f"within-image  μ={within_arr.mean():.3f}", density=True)
axes[0].hist(across_arr, bins=80, alpha=0.7, color="#888888",
             label=f"random pairs  μ={across_arr.mean():.3f}", density=True)
axes[0].set_xlabel("Pairwise cosine similarity of conditions")
axes[0].set_ylabel("Density")
axes[0].set_title("2a — Within-image vs. random condition similarity")
axes[0].legend()

# 2b: caption diversity vs. condition diversity per image
img_cond_var, img_wc_var = [], []
for img_id in multi_cap_ids[:10000]:
    idxs = img_id_to_idxs[img_id]
    c    = all_conds_np[idxs]
    img_cond_var.append(c.var(axis=0).mean())
    wcs = np.array([len(train_captions[i].split()) for i in idxs])
    img_wc_var.append(float(wcs.var()))

img_cond_var = np.array(img_cond_var)
img_wc_var   = np.array(img_wc_var)
r_wc = np.corrcoef(img_wc_var, img_cond_var)[0, 1]

axes[1].scatter(img_wc_var, img_cond_var, s=3, alpha=0.2)
axes[1].set_xlabel("Caption word-count variance within image")
axes[1].set_ylabel("Mean condition variance within image")
axes[1].set_title(f"2b — Caption diversity vs condition diversity  (r={r_wc:.3f})")

plt.tight_layout()
plt.show()

---
## 3 — Caption Property ↔ Condition Correlation

**What this shows.** Without type labels, we ask: *which measurable linguistic properties of
a caption predict where its condition lands in condition space?*

Properties computed per training caption:
- **Word count** — caption length
- **Avg word length** — proxy for vocabulary complexity
- **Lexical diversity (TTR)** — type-token ratio; higher = less repetition
- **Digit presence** — captions mentioning counts or numbers

Each property is correlated with the top-3 PCs of the condition space.
The scatter panel shows the strongest correlate.

In [ ]:
print("Computing caption properties for all training samples...")
wc_all, awl_all, ttr_all, dig_all = [], [], [], []
for cap in train_captions:
    words = cap.lower().split()
    wc_all.append(len(words))
    awl_all.append(np.mean([len(w) for w in words]) if words else 0.0)
    ttr_all.append(len(set(words)) / len(words) if words else 0.0)
    dig_all.append(float(any(ch.isdigit() for ch in cap)))

prop_names = ["Word count", "Avg word length", "Lexical diversity (TTR)", "Has digit"]
prop_mat   = np.stack([wc_all, awl_all, ttr_all, dig_all], axis=1)  # [N, 4]

# PCA on full training conditions
pca3 = PCA(n_components=3, random_state=42)
conds_pcs = pca3.fit_transform(all_conds_np)   # [N, 3]
var_exp = pca3.explained_variance_ratio_ * 100
print(f"PC1={var_exp[0]:.1f}%  PC2={var_exp[1]:.1f}%  PC3={var_exp[2]:.1f}%")

In [ ]:
# Correlation matrix: [3 PCs] × [4 properties]
full_mat  = np.concatenate([conds_pcs, prop_mat], axis=1)   # [N, 7]
corr_full = np.corrcoef(full_mat.T)                          # [7, 7]
cross_corr = corr_full[:3, 3:]                               # [3, 4]  PCs vs props

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
im = axes[0].imshow(cross_corr, cmap="RdBu_r", vmin=-0.6, vmax=0.6, aspect="auto")
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(prop_names, rotation=30, ha="right")
axes[0].set_yticks(range(3))
axes[0].set_yticklabels([f"PC{i+1} ({var_exp[i]:.1f}%)" for i in range(3)])
for i in range(3):
    for j in range(4):
        axes[0].text(j, i, f"{cross_corr[i,j]:.3f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=axes[0])
axes[0].set_title("Condition PC ↔ Caption property correlation")

# Scatter for the single strongest correlate
bi, bj = np.unravel_index(np.abs(cross_corr).argmax(), cross_corr.shape)
samp   = rng.choice(len(conds_pcs), 12000, replace=False)
axes[1].scatter(prop_mat[samp, bj], conds_pcs[samp, bi], s=2, alpha=0.15)
axes[1].set_xlabel(prop_names[bj])
axes[1].set_ylabel(f"PC{bi+1}")
axes[1].set_title(f"Strongest correlate  r={cross_corr[bi, bj]:.3f}")

plt.tight_layout()
plt.show()

---
## 4 — Ablation: Retrieval Quality under Different Conditions

**What this shows.** How much does the specific condition value matter, vs. the combiner
transformation alone?

For the img-side combiner, we apply one condition to the **entire test image gallery** (5000 images),
then retrieve all 25000 test texts against it.

Conditions tested:
- **CLIP baseline** — raw CLIP embeddings, no combiner
- **Zero condition** — combiner active, but with a zero vector → tests the learned mapping alone
- **Random condition** — random vector at the median training condition norm
- **Mean of all training conditions** — the "average" condition
- **All K representatives** — the K learned prototype conditions

The per-sample rank-delta histogram (bottom) shows the distribution of improvement over CLIP
for the single best representative.

In [ ]:
rng_t = torch.Generator(); rng_t.manual_seed(0)
random_cond = torch.randn(label_dim, generator=rng_t)
random_cond = random_cond / random_cond.norm() * median_norm
mean_cond   = label_emb_all.mean(0)

named_conds = [
    ("CLIP baseline",     None),
    ("Zero condition",    torch.zeros(label_dim)),
    ("Random condition",  random_cond),
    ("Mean of all train", mean_cond),
]
for k in range(K_reps):
    named_conds.append((f"Rep {k:02d}", representatives[k]))

results = {}
for name, cond in named_conds:
    print(f"  {name:<22} ...", end="  ")
    m = retrieval_metrics(cond)
    results[name] = m
    print(f"R@1={m['R@1']:5.1f}%  R@5={m['R@5']:5.1f}%  R@10={m['R@10']:5.1f}%  MR={m['mean_rank']:6.1f}")

# Identify best / median / worst representative
rep_r1 = sorted(
    [(k, results[f"Rep {k:02d}"]['R@1']) for k in range(K_reps)],
    key=lambda x: x[1]
)
best_rep_k   = rep_r1[-1][0]
median_rep_k = rep_r1[len(rep_r1) // 2][0]
worst_rep_k  = rep_r1[0][0]
print(f"\nBest  rep: Rep {best_rep_k:02d}  R@1={rep_r1[-1][1]:.1f}%")
print(f"Median rep: Rep {median_rep_k:02d}  R@1={rep_r1[len(rep_r1)//2][1]:.1f}%")
print(f"Worst rep: Rep {worst_rep_k:02d}  R@1={rep_r1[0][1]:.1f}%")

In [ ]:
# Bar chart: key conditions only
key_names = [
    "CLIP baseline", "Zero condition", "Random condition", "Mean of all train",
    f"Rep {best_rep_k:02d} (best)",
    f"Rep {median_rep_k:02d} (median)",
    f"Rep {worst_rep_k:02d} (worst)",
]
key_results = {
    "CLIP baseline":        results["CLIP baseline"],
    "Zero cond":            results["Zero condition"],
    "Random cond":          results["Random condition"],
    "Mean cond":            results["Mean of all train"],
    f"Rep {best_rep_k:02d} (best)":   results[f"Rep {best_rep_k:02d}"],
    f"Rep {median_rep_k:02d} (med)": results[f"Rep {median_rep_k:02d}"],
    f"Rep {worst_rep_k:02d} (worst)": results[f"Rep {worst_rep_k:02d}"],
}

metrics_show = ["R@1", "R@5", "R@10"]
x     = np.arange(len(key_results))
width = 0.25
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, metric in enumerate(metrics_show):
    vals = [r[metric] for r in key_results.values()]
    axes[0].bar(x + i * width, vals, width, label=metric)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(list(key_results.keys()), rotation=35, ha="right", fontsize=8)
axes[0].set_ylabel("%"); axes[0].set_title("Retrieval quality under different conditions")
axes[0].legend()

# Per-sample rank delta: CLIP vs best representative
baseline_ranks = results["CLIP baseline"]["ranks"]
best_ranks     = results[f"Rep {best_rep_k:02d}"]["ranks"]
delta = baseline_ranks.astype(float) - best_ranks.astype(float)
pct_improved   = (delta > 0).mean() * 100
pct_degraded   = (delta < 0).mean() * 100

axes[1].hist(delta, bins=80, color="#2196F3", alpha=0.8)
axes[1].axvline(0,          color="k",   linestyle="--", linewidth=1)
axes[1].axvline(delta.mean(), color="r", linestyle="--",
                label=f"mean Δ={delta.mean():.1f}")
axes[1].set_xlabel("Rank improvement  (CLIP rank − conditioned rank)")
axes[1].set_ylabel("Query count")
axes[1].set_title(f"Per-query rank delta (best rep vs CLIP)\n"
                  f"{pct_improved:.1f}% improved · {pct_degraded:.1f}% degraded")
axes[1].legend()
plt.tight_layout(); plt.show()

# Show captions of top-10 most improved and most degraded queries
top_improved = np.argsort(delta)[::-1][:10]
top_degraded = np.argsort(delta)[:10]
print("\n── Top-10 most improved queries ──")
for i, q in enumerate(top_improved):
    print(f"  [{i+1}] Δrank={delta[q]:+.0f}  '{all_raw_text[q]}'")
print("\n── Top-10 most degraded queries ──")
for i, q in enumerate(top_degraded):
    print(f"  [{i+1}] Δrank={delta[q]:+.0f}  '{all_raw_text[q]}'")

---
## 5 — Directional Sweep Along PCA Axes

**What this shows.** Replaces the type-mean directional sweep from the impressions notebook.
We identify the top-2 PCA directions of the training condition space and sweep a probe
condition along each, from −2σ to +2σ.

For the img-side combiner, the swept condition is applied to the full test image gallery,
and we show the top-retrieved image for a fixed text query at each step.
A green border marks steps where the ground-truth image is in the top-5.

If sweeping along PC1 produces a coherent visual progression (e.g., indoor → outdoor,
or specific → generic scenes), that direction encodes a semantically interpretable axis.

In [ ]:
from sklearn.decomposition import PCA as _PCA

pca_dir = _PCA(n_components=3, random_state=42)
pca_dir.fit(all_conds_np)
directions = {
    f"PC1 ({pca_dir.explained_variance_ratio_[0]*100:.1f}%)": torch.tensor(pca_dir.components_[0], dtype=torch.float32),
    f"PC2 ({pca_dir.explained_variance_ratio_[1]*100:.1f}%)": torch.tensor(pca_dir.components_[1], dtype=torch.float32),
}

query_txt   = all_raw_text[SWEEP_QUERY_IDX]
gt_img_idx  = int(txt2img[SWEEP_QUERY_IDX])
query_txt_n = txt_n[SWEEP_QUERY_IDX]    # [512]
print(f"Query text: '{query_txt}'")
print(f"GT image  : {image_paths[gt_img_idx]}")

sweep_ts = np.linspace(-2.0, 2.0, N_SWEEP)

for dir_name, direction in directions.items():
    proj_std = float((label_emb_all @ direction).std())

    fig, axes = plt.subplots(1, N_SWEEP, figsize=(N_SWEEP * 1.6, 2.2))
    fig.suptitle(f"Sweep along {dir_name}  |  query: '{query_txt[:55]}'")

    for j, t in enumerate(sweep_ts):
        cond = direction * (t * proj_std)
        cond_gallery_n = F.normalize(apply_condition(all_img_emb, cond), dim=-1)
        scores_q = query_txt_n @ cond_gallery_n.T    # [5000]
        top5     = scores_q.topk(N_RETRIEVE).indices.tolist()
        top1_path = image_paths[top5[0]]
        gt_in_top5 = gt_img_idx in top5

        show_img(axes[j], top1_path,
                 border_color="green" if gt_in_top5 else None)
        axes[j].set_title(f"{t:+.1f}σ", fontsize=7)

    plt.tight_layout()
    plt.show()

---
## 6 — Epoch Evolution (Geometry-Based)

**What this shows.** How the condition space changes over training — without needing type labels.

Tracked per checkpoint:
- **Condition norm** (mean ± std) — does the model push conditions to larger magnitudes?
- **Within-image similarity** — does the model progressively converge conditions for the
  same image together? A rising curve means the model is learning image-level conditions.
  A flat or noisy curve suggests conditions remain caption-specific throughout.

This replaces the "epoch evolution of type-mean trajectories" from the impressions notebook.

In [ ]:
# Sample image ids for within-image analysis (speed)
sampled_img_ids = multi_cap_ids[:N_WITHIN_SAMPLE]

epoch_stats = []
for f in epoch_files:
    snap_e = torch.load(f, map_location="cpu", weights_only=False)
    ep     = snap_e["epoch"]
    conds_e = snap_e["label_embeddings_all"].numpy()   # [566747, 16]
    norms_e = np.linalg.norm(conds_e, axis=1)
    conds_e_n = conds_e / (norms_e[:, None] + 1e-8)

    w_sims = []
    for img_id in sampled_img_ids:
        idxs = img_id_to_idxs[img_id]
        c    = conds_e_n[idxs]
        sim  = c @ c.T
        n    = len(idxs)
        w_sims.extend(sim[i, j] for i in range(n) for j in range(i + 1, n))

    epoch_stats.append({
        "epoch":          ep,
        "norm_mean":      norms_e.mean(),
        "norm_std":       norms_e.std(),
        "within_mean":    np.mean(w_sims),
        "within_std":     np.std(w_sims),
    })
    print(f"  epoch {ep:3d} | norm {norms_e.mean():.4f}±{norms_e.std():.4f} "
          f"| within_sim {np.mean(w_sims):.4f}±{np.std(w_sims):.4f}")

In [ ]:
epochs_x     = [s["epoch"]      for s in epoch_stats]
norm_means   = [s["norm_mean"]  for s in epoch_stats]
norm_stds    = [s["norm_std"]   for s in epoch_stats]
within_means = [s["within_mean"] for s in epoch_stats]
within_stds  = [s["within_std"]  for s in epoch_stats]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_x, norm_means, "o-", color="#FF9800")
axes[0].fill_between(epochs_x,
    np.array(norm_means) - np.array(norm_stds),
    np.array(norm_means) + np.array(norm_stds),
    alpha=0.2, color="#FF9800")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Condition L2 norm")
axes[0].set_title("Condition norm over training")

axes[1].plot(epochs_x, within_means, "o-", color="#2196F3", label="within-image sim")
axes[1].fill_between(epochs_x,
    np.array(within_means) - np.array(within_stds),
    np.array(within_means) + np.array(within_stds),
    alpha=0.2, color="#2196F3")
axes[1].axhline(float(across_arr.mean()), color="gray", linestyle="--",
                label=f"random pairs baseline ({across_arr.mean():.3f})")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Cosine similarity")
axes[1].set_title("Within-image condition similarity over training\n"
                  "(rising = model learns image-level conditions)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 7 — Per-Image Deep Dive

**What this shows.** A qualitative look at individual test images.

For each chosen test image we show:
- All 5 captions as query texts.
- For each caption: CLIP baseline retrieval vs best-representative-condition retrieval,
  side by side. Green border = ground-truth image appears in the top-5.
- A bar chart comparing cosine similarity of each caption to the GT image,
  before and after applying the best representative condition.

In [ ]:
best_cond       = representatives[best_rep_k]   # [16]
best_gallery_n  = F.normalize(apply_condition(all_img_emb, best_cond), dim=-1)   # [5000, 512]


def deep_dive(img_idx: int):
    cap_indices = img2txt[img_idx].tolist()         # 5 text indices
    captions    = [all_raw_text[i] for i in cap_indices]
    gt_path     = image_paths[img_idx]
    n_caps      = len(cap_indices)

    fig, axes = plt.subplots(
        n_caps, 1 + N_RETRIEVE * 2,
        figsize=((1 + N_RETRIEVE * 2) * 1.4, n_caps * 1.6),
        gridspec_kw={"width_ratios": [2.5] + [1] * (N_RETRIEVE * 2)}
    )
    if n_caps == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f"Test image {img_idx}  —  GT: {os.path.basename(gt_path)}",
                 fontsize=10)

    clip_sims, cond_sims = [], []

    for row, (cap_idx, caption) in enumerate(zip(cap_indices, captions)):
        q_n = txt_n[cap_idx]   # [512]

        # CLIP scores
        clip_scores = q_n @ img_n.T        # [5000]
        clip_top5   = clip_scores.topk(N_RETRIEVE).indices.tolist()
        clip_gt_sim = float(clip_scores[img_idx])

        # Conditioned scores
        cond_scores = q_n @ best_gallery_n.T   # [5000]
        cond_top5   = cond_scores.topk(N_RETRIEVE).indices.tolist()
        cond_gt_sim = float(cond_scores[img_idx])

        clip_sims.append(clip_gt_sim)
        cond_sims.append(cond_gt_sim)

        clip_rank = int((clip_scores > clip_scores[img_idx]).sum()) + 1
        cond_rank = int((cond_scores > cond_scores[img_idx]).sum()) + 1

        # Caption label column
        axes[row, 0].axis("off")
        axes[row, 0].text(
            0.5, 0.5,
            f"{caption[:55]}\n"
            f"CLIP rank {clip_rank} → cond rank {cond_rank}",
            ha="center", va="center", fontsize=6, wrap=True,
            transform=axes[row, 0].transAxes
        )

        # CLIP top-5
        for j, ret_idx in enumerate(clip_top5):
            show_img(axes[row, 1 + j], image_paths[ret_idx],
                     border_color="green" if ret_idx == img_idx else None)

        # Conditioned top-5
        for j, ret_idx in enumerate(cond_top5):
            show_img(axes[row, 1 + N_RETRIEVE + j], image_paths[ret_idx],
                     border_color="green" if ret_idx == img_idx else None)

    # Column headers
    axes[0, 1].set_title("CLIP baseline →", fontsize=7)
    axes[0, 1 + N_RETRIEVE].set_title(f"Conditioned (rep {best_rep_k:02d}) →", fontsize=7)

    plt.tight_layout()
    plt.show()

    # Similarity bar chart
    fig2, ax2 = plt.subplots(figsize=(8, 3))
    x2 = np.arange(n_caps)
    ax2.bar(x2 - 0.2, clip_sims, 0.4, label="CLIP baseline", color="#888888")
    ax2.bar(x2 + 0.2, cond_sims, 0.4, label=f"Rep {best_rep_k:02d} condition", color="#2196F3")
    ax2.set_xticks(x2)
    ax2.set_xticklabels([f"Cap {i}" for i in range(n_caps)])
    ax2.set_ylabel("Cosine sim to GT image")
    ax2.set_title(f"Image {img_idx} — GT image similarity before/after conditioning")
    ax2.legend()
    plt.tight_layout()
    plt.show()


for img_idx in DEEP_DIVE_IMGS:
    deep_dive(img_idx)